# Cadrage : entraîner la LoRA de Qwen3-4B

Tu vas entraîner un petit « supplément » (une LoRA) sur environ 340 exemples publics (rapports de la Cour des comptes européenne et pièces de marché), puis fabriquer
**deux modèles** pour Ollama, passés par exactement la même chaîne : la **base** et la **base + LoRA**.
La LoRA apprend le **format**, la citation et le refus. Elle n'apprend aucun fait de client.

**Avant tout :** menu **Exécution → Modifier le type d'exécution → T4 GPU → Enregistrer**.

Lance les cellules **une par une, dans l'ordre** (pas « Tout exécuter »). Chaque titre dit l'étape et sa durée
estimée (estimations, rien n'a encore tourné sur Colab).

| Étape | Durée estimée |
|---|---|
| 1 à 7 : préparer (à refaire à chaque nouvelle session Colab) | 20 à 30 min |
| Partie 0 : preuve | 15 à 25 min |
| Parties 1, 2 et 3 : entraînement | 1 h 30 à 3 h en tout (non mesuré) |
| Exporter le modèle entraîné, puis la base | 20 à 30 min chacun |
| Résumé | 1 min |
| Test avant téléchargement : base contre LoRA, par tâche | 40 à 80 min |

Si Colab coupe : relance les étapes 1 à 7, puis la **même** partie. Tout est sauvegardé sur ton Drive.
Aucun mot de passe ni jeton n'est demandé : tu autorises seulement l'accès à ton Drive quand Colab le propose.
Données : uniquement des documents publics (CC BY 4.0, Licence Ouverte 2.0). **Jamais de document client.**

In [ ]:
#@title Étape 1 sur 7 : brancher le GPU et ton Google Drive (1 min) {display-mode: "form"}
import hashlib, os, shutil, signal, subprocess, sys, time, urllib.request, zipfile
from pathlib import Path

CODE = "e20ddf3a3a74fb674796f6121b47d32493cd0cac"            # version exacte du code
LLAMA = "e300919cc704ddde3a8a8346a8714f57eeeee71b"   # version exacte de llama.cpp
DONNEES = "5909f70d2ff0b5279b2637712e0039c282af857381f02812b26cf2b236a32a26"      # empreinte SHA-256 du paquet de données publiques
OLLAMA = "0.40.17"         # la même version que sur le PC
BASE_REVISION = "1cfa9a7208912126459214e8b04321603b3df60c"   # Qwen/Qwen3-4B, révision épinglée

def sh(commande):
    r = subprocess.run(["bash", "-c", "set -o pipefail; " + commande], capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"Échec : {commande}\n{(r.stderr or r.stdout)[-1500:]}")
    return r.stdout

if subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode:
    raise RuntimeError("Pas de GPU. Menu Exécution, Modifier le type d'exécution, T4 GPU, puis relance.")
print("GPU :", sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader").strip())

from google.colab import drive
drive.mount("/content/drive")
STOCK = Path("/content/drive/MyDrive/cadrage-entrainement")
STOCK.mkdir(parents=True, exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"Drive branché. Place libre : {libre:.0f} Go (il en faut environ 8).")
if libre < 8:
    raise RuntimeError("Pas assez de place sur ton Drive (8 Go minimum). Fais de la place puis relance.")

In [ ]:
#@title Étape 2 sur 7 : récupérer le code et les données publiques, avec vérification (2 min) {display-mode: "form"}
GITHUB = "https://github.com/YoussefBY22/PRV-PR8-PFE-IA-Cadrage"
RAW = "https://raw.githubusercontent.com/YoussefBY22/PRV-PR8-PFE-IA-Cadrage"

contenu = urllib.request.urlopen(f"{RAW}/{CODE}/training/colab/cadrage-donnees-publiques.zip", timeout=120).read()
if hashlib.sha256(contenu).hexdigest() != DONNEES:
    raise RuntimeError("Le paquet de données ne correspond pas à l'empreinte attendue : n'entraîne pas, préviens Claude.")
ARCHIVE = STOCK / "cadrage-donnees-publiques.zip"
ARCHIVE.write_bytes(contenu)
print("Paquet de données vérifié (SHA-256 identique) :", DONNEES[:16], "...")

os.chdir("/content")
REPO = Path("/content/cadrage")
sh(f"rm -rf {REPO} && git init -q {REPO} && git -C {REPO} fetch -q --depth 1 {GITHUB}.git {CODE} && git -C {REPO} checkout -q FETCH_HEAD")
zipfile.ZipFile(ARCHIVE).extractall(REPO / "training/data/real")
import json
m = json.loads((REPO / "training/data/real/MANIFEST.json").read_text(encoding="utf-8"))
for part in ("train", "val", "test"):
    e = m["effectifs"][part]
    print(f"{part:5s} : {e['exemples']:2d} exemples, {e['documents']} documents")
print("Licences :", " ; ".join(m["licences"]))

In [ ]:
#@title Étape 3 sur 7 : installer les bibliothèques d'entraînement (3 min) {display-mode: "form"}
sh(f"{sys.executable} -m pip install -q transformers==4.57.6 peft==0.17.1 accelerate==1.10.1 "
   "bitsandbytes==0.50.2 huggingface_hub==0.36.2 sentencepiece pyyaml")
print("Bibliothèques installées.")

In [ ]:
#@title Étape 4 sur 7 : télécharger Qwen3-4B, version épinglée (8 Go, 5 à 10 min) {display-mode: "form"}
BASE = sh(f"""{sys.executable} -c 'from huggingface_hub import snapshot_download as d; """
          f"""print(d("Qwen/Qwen3-4B", revision="{BASE_REVISION}"))'""").strip().splitlines()[-1]
print("Modèle de base prêt :", BASE)

In [ ]:
#@title Étape 5 sur 7 : installer Ollama et vérifier le gabarit de qwen3:4b (5 min) {display-mode: "form"}
if not shutil.which("ollama"):
    sh("apt-get install -y -qq zstd")
    sh(f"curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA} sh")
def ollama_pret():
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        return True
    except OSError:
        return False
if not ollama_pret():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(60):
        if ollama_pret():
            break
        time.sleep(1)
sh("ollama pull qwen3:4b")
print(sh(f"cd {REPO} && {sys.executable} -c 'from training import export_gguf as e; e.check_template(); print(\"Gabarit de qwen3:4b identique à celui du dépôt.\")'").strip())

In [ ]:
#@title Étape 6 sur 7 : compiler l'outil de quantification de llama.cpp (5 min) {display-mode: "form"}
sh("rm -rf /content/llama.cpp && git init -q /content/llama.cpp && git -C /content/llama.cpp fetch -q --depth 1 "
   f"https://github.com/ggml-org/llama.cpp.git {LLAMA} && git -C /content/llama.cpp checkout -q FETCH_HEAD")
sh(f"{sys.executable} -m pip install -q numpy sentencepiece protobuf")
print(sh(f"cd {REPO} && {sys.executable} -c 'from pathlib import Path; from training import export_gguf as e; print(e.build_quantize(Path(\"/content/llama.cpp\")))'").strip())

In [ ]:
#@title Étape 7 sur 7 : outil de lancement (instantané) {display-mode: "form"}
def cadrage(*action):
    env = dict(os.environ, LLAMA_CPP_DIR="/content/llama.cpp", PYTHONUNBUFFERED="1",
               HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    cmd = [sys.executable, "-m", "training.parts", *map(str, action),
           "--store", str(STOCK / "etat-4b"), "--base-dir", BASE]
    p = subprocess.Popen(cmd, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors="replace", start_new_session=True)
    try:
        for ligne in p.stdout:
            print(ligne, end="")
    except KeyboardInterrupt:
        print("Arrêt demandé : sauvegarde en cours, attends le message de fin.")
        p.send_signal(signal.SIGINT)
        for ligne in p.stdout:
            print(ligne, end="")
    code = p.wait()
    os.sync()
    if code:
        raise RuntimeError(f"Non terminé (code {code}). Copie ce qui est affiché au-dessus à Claude.")

cadrage("etat")
print("PRÊT. Tu peux lancer la partie 0.")

In [ ]:
#@title Partie 0 : preuve que tout fonctionne (15 à 25 min, obligatoire) {display-mode: "form"}
# 2 pas d'entraînement, sauvegarde, reprise, export complet (GGUF Q4_K_M) et réponse d'Ollama.
# Si elle échoue : ne va pas plus loin, copie le message à Claude.
cadrage("preuve")

In [ ]:
#@title Partie 1 : entraînement, premier tiers (30 à 60 min) {display-mode: "form"}
# Si Colab coupe : relance les étapes 1 à 7 puis CETTE cellule : elle reprend où elle s'était arrêtée.
cadrage("partie", 1)

In [ ]:
#@title Partie 2 : entraînement, deuxième tiers (30 à 60 min) {display-mode: "form"}
# Si l'erreur de validation remontait déjà (arrêt anticipé), cette cellule le dit et ne fait rien : c'est normal.
cadrage("partie", 2)

In [ ]:
#@title Partie 3 : entraînement, dernière partie (30 à 60 min) {display-mode: "form"}
# Si l'erreur de validation remontait déjà (arrêt anticipé), cette cellule le dit et ne fait rien : c'est normal.
cadrage("partie", 3)

In [ ]:
#@title Où j'en suis (instantané, à lancer quand tu veux) {display-mode: "form"}
cadrage("etat")

In [ ]:
#@title Exporter le modèle entraîné : fusion, GGUF, Q4_K_M, SHA-256 (15 à 30 min, 2,5 Go sur Drive) {display-mode: "form"}
cadrage("exporter")

In [ ]:
#@title Exporter la base Qwen3-4B par la MÊME chaîne, pour comparer honnêtement (15 à 30 min, 2,5 Go) {display-mode: "form"}
cadrage("exporter-base")

In [ ]:
#@title Résumé : pas, pertes, durée, fichiers à télécharger (instantané) {display-mode: "form"}
cadrage("resume")
print()
print("TÉLÉCHARGE depuis ton Drive, dossier cadrage-entrainement/etat-4b/modeles :")
print("  cadrage-4b-lora/model.gguf et cadrage-4b-lora/model.gguf.sha256")
print("  cadrage-4b-base/model.gguf et cadrage-4b-base/model.gguf.sha256")
print("  SHA256SUMS.txt")
print("Puis, sur ton PC : python -m training.import_ollama --lora <dossier>/cadrage-4b-lora --base <dossier>/cadrage-4b-base")

In [ ]:
#@title Test avant téléchargement : base contre LoRA, par type de tâche (20 à 60 min) {display-mode: "form"}
# À lancer APRÈS les deux exports. Rien n'est téléchargé : les deux modèles sont déjà dans l'Ollama de Colab.
# Les exemples de test viennent de rapports jamais vus à l'entraînement. Mesure par type de tâche (meta.task_type) :
# JSON valide, bonne décision (répondre ou refuser), phrases vérifiées par le vérificateur de l'application,
# et pour les tâches « position » et « conflit », étiquette (accept / partial / reject ; conflict / aligned) notée
# comme une classification contre la position officielle de la Commission ; pour « axes », paires de constats
# rangées ensemble ou séparément comme dans le sommaire du rapport (les noms d'axes peuvent différer).
import json, re, sys, time, urllib.request, zipfile
from collections import Counter, defaultdict
sh(f"{sys.executable} -m pip install -q snowballstemmer pydantic")
sys.path.insert(0, "/content/cadrage/backend")
from app.domain.documents import Chunk
from app.domain.generation import SectionDraft, Sentence
from app.generation.llm_schemas import LLMFindings, LLMSection, LLMStance, build_axes_schema
from app.verification import verify_section

z = zipfile.ZipFile(STOCK / "cadrage-donnees-publiques.zip")
nom = next(n for n in z.namelist() if n.endswith("test.jsonl"))
tests = [json.loads(l) for l in z.read(nom).decode("utf-8").splitlines() if l.strip()]
TACHES = sorted({e["meta"].get("task_type", "autre") for e in tests})
print(f"{len(tests)} exemples de test :", dict(Counter(e["meta"].get("task_type", "autre") for e in tests)))

def schema_de(ex):
    t = ex["meta"]["rendu"]["tache"]
    if t == "axes":
        r = ex["meta"]["rendu"]
        return build_axes_schema([f"F{i}" for i in range(1, len(r["constats"]) + 1)], len(r["axes_connus"]))
    return LLMFindings if t == "constats" else LLMStance if t in ("position", "conflict") else LLMSection

def demander(modele, messages, schema):
    corps = {"model": modele, "messages": messages, "stream": False, "think": False,
             "format": schema.model_json_schema(), "truncate": False, "shift": False,
             "options": {"temperature": 0, "num_ctx": 4096}}
    req = urllib.request.Request("http://127.0.0.1:11434/api/chat", json.dumps(corps).encode(),
                                 {"Content-Type": "application/json"})
    t = time.time()
    r = json.loads(urllib.request.urlopen(req, timeout=600).read())
    return r["message"]["content"], time.time() - t

def groupes(ex, reponse):
    """Axe de chaque constat F1..Fn (identifiant), d'après une réponse LLMAxes."""
    par_f = {x.get("finding"): x.get("axis") for x in reponse.get("assignments", [])}
    return [par_f.get(f"F{i}") for i in range(1, len(ex["meta"]["rendu"]["constats"]) + 1)]

def noter_axes(ex, texte):
    attendu = groupes(ex, json.loads(ex["messages"][2]["content"]))
    try:
        obtenu = groupes(ex, json.loads(texte))
    except ValueError:
        return {"json": 0}
    paires = [(i, j) for i in range(len(attendu)) for j in range(i + 1, len(attendu))]
    ok = sum((attendu[i] == attendu[j]) == (obtenu[i] is not None and obtenu[i] == obtenu[j]) for i, j in paires)
    return {"json": 1, "statut_juste": int(None not in obtenu), "phrases": 0, "verifiees": 0, "bloquantes": 0,
            "francais": 1, "paires_justes": ok, "paires_n": len(paires),
            "axes_attendus": len(set(attendu)), "axes_obtenus": len(set(x for x in obtenu if x))}

def noter(ex, texte):
    if ex["meta"]["rendu"]["tache"] == "axes":
        return noter_axes(ex, texte)
    passages = dict(re.findall(r"<<PASSAGE id=(P\d+)[^>]*>>\n(.*?)\n<<FIN", ex["messages"][1]["content"], re.S))
    attendu = json.loads(ex["messages"][2]["content"])
    try:
        sortie = json.loads(texte)
    except ValueError:
        return {"json": 0}
    constats = "findings" in attendu
    phrases = sortie.get("findings" if constats else "sentences") or []
    refus_attendu = attendu.get("status") == "insufficient_evidence"
    refus = sortie.get("status") == "insufficient_evidence" or not phrases
    chunks = {l: Chunk(id=l, document_id="d", dossier_id="d", ordinal=0, text=t, page_start=1, page_end=1,
                       section_path=(), token_estimate=1) for l, t in passages.items()}
    ss = [Sentence(text=p.get("text", ""), kind=p.get("kind", "fact"),
                   source_ids=[re.sub(r"\W", "", s).upper() for s in p.get("sources", [])]) for p in phrases]
    rep = verify_section(SectionDraft(section_key="t", status="ok", sentences=ss), chunks, dossier_id="d")
    note = {"json": 1, "statut_juste": int(refus == refus_attendu), "phrases": len(ss),
            "verifiees": sum(c.status == "verified" for c in rep.checks),
            "bloquantes": sum(i.severity == "blocking" for i in rep.issues),
            "francais": int(all(not re.search(r"\b(the|and|of|with)\b", s.text) for s in ss))}
    if ex["meta"].get("etiquette"):  # position / conflit : classification
        note["etiquette_juste"] = int(sortie.get("stance") == ex["meta"]["etiquette"])
        note["etiquette_n"] = 1
        note["conf"] = f'{ex["meta"]["etiquette"]}->{sortie.get("stance")}'
    return note

resultats, confusions = {}, defaultdict(Counter)
for modele in ("cadrage-4b-base", "cadrage-4b-lora"):
    par_tache, temps = defaultdict(Counter), 0.0
    for i, ex in enumerate(tests, 1):
        texte, s = demander(modele, ex["messages"][:2], schema_de(ex))
        note = noter(ex, texte)
        tache = ex["meta"].get("task_type", "autre")
        par_tache[tache].update({k: v for k, v in note.items() if isinstance(v, int)})
        par_tache[tache]["n"] += 1
        if "conf" in note:
            confusions[modele][note["conf"]] += 1
        temps += s
        print(f"{modele} {i}/{len(tests)} {tache} {note}")
    resultats[modele] = (par_tache, temps)

print("\n================ RÉSUMÉ PAR TÂCHE (à envoyer à Claude) ================")
for modele, (par_tache, temps) in resultats.items():
    print(f"\n### {modele} ({temps / len(tests):.0f} s par exemple)")
    total = Counter()
    for tache in TACHES:
        t = par_tache[tache]
        total.update(t)
        et = f" | étiquette juste {t['etiquette_juste']}/{t['etiquette_n']}" if t["etiquette_n"] else ""
        if t["paires_n"]:
            et += (f" | paires bien rangées {t['paires_justes']}/{t['paires_n']} | axes obtenus "
                   f"{t['axes_obtenus']} pour {t['axes_attendus']} attendus")
        print(f"{tache:18s} n={t['n']:3d} | JSON {t['json']}/{t['n']} | statut juste {t['statut_juste']}/{t['n']} | "
              f"phrases vérifiées {t['verifiees']}/{t['phrases']} | bloquantes {t['bloquantes']}{et}")
    print(f"{'TOTAL':18s} n={total['n']:3d} | JSON {total['json']}/{total['n']} | statut juste {total['statut_juste']}/{total['n']} | "
          f"phrases vérifiées {total['verifiees']}/{total['phrases']} | bloquantes {total['bloquantes']} | "
          f"tout en français {total['francais']}/{total['n']}")
    if confusions[modele]:
        print("confusions (attendu->obtenu) :", dict(confusions[modele]))

ex = tests[0]
print("\n================ EXEMPLE : même question, deux modèles ================")
for modele in resultats:
    print(f"\n--- {modele}\n", demander(modele, ex["messages"][:2], schema_de(ex))[0][:1500])
print("\n--- réponse humaine attendue\n", ex["messages"][2]["content"][:1500])